---
short_title: Einlesen und Filtern
numbering:
    heading_1: true
    heading_2: true
    title: true
---
# MARC-XML einlesen und filtern

```{caution} Work In Progress
Diese Inhalte sind noch in Bearbeitung.
```

In diesem Notebook lesen wir die synthetische {term}`MARC-XML`-Datei `beispieldaten.marcxml` mit
der Standardbibliothek [`xml.etree.ElementTree`](https://docs.python.org/3/library/xml.etree.elementtree.html)
ein. Anschließend lesen wir Felder und Unterfelder aus und filtern die Titel
nach Erscheinungsjahr, Sprache und Schlagwort.


## Den Datenpfad finden

Ob ein Notebook aus dem Projektverzeichnis oder aus dem Kapitelverzeichnis
gestartet wird, ist nicht immer gleich. Damit unsere Beispiele zuverlässig
laufen, suchen wir das Verzeichnis, das den Ordner `assets/data` enthält, und
arbeiten von dort aus mit relativen Pfaden weiter.

```{note}
Diese kleine Hilfsfunktion werden Sie in den folgenden Notebooks
wiederfinden. Sie ist kein MARC-Thema, sondern eine praktische
Pfad-Konvention.
```


In [1]:
from pathlib import Path


def finde_projektwurzel():
    """Sucht das Verzeichnis, das den Ordner assets/data enthält."""
    start = Path.cwd().resolve()
    for kandidat in [start, *start.parents]:
        if (kandidat / "assets" / "data").is_dir():
            return kandidat
    raise FileNotFoundError("Projektwurzel mit assets/data nicht gefunden.")


PROJEKTWURZEL = finde_projektwurzel()
DATEN = PROJEKTWURZEL / "assets" / "data"
MARCXML = DATEN / "beispieldaten.marcxml"
print(MARCXML.relative_to(PROJEKTWURZEL))

assets/data/beispieldaten.marcxml


## Die {term}`XML`-Datei einlesen

`ElementTree` liest die Datei mit `ET.parse()` ein und liefert einen Baum, an
dessen Spitze das Wurzelelement steht. Beachten Sie die Ausgabe: Vor dem
Elementnamen steht der **Namensraum** in geschweiften Klammern. Wir hinterlegen
ihn deshalb einmal als Kürzel `m` (für MARC) in einem Dictionary.


In [2]:
from xml.etree import ElementTree as ET

MARC_NS = "http://www.loc.gov/MARC21/slim"
NAMENSRAUM = {"m": MARC_NS}

baum = ET.parse(MARCXML)
wurzel = baum.getroot()
print(wurzel.tag)

{http://www.loc.gov/MARC21/slim}collection


## Records auflisten

Die Sammlung (`collection`) enthält die einzelnen Records. Mit `findall()` und
dem Kürzel `m:record` erhalten wir alle direkten Kind-Elemente, die Records
sind.


In [3]:
records = wurzel.findall("m:record", NAMENSRAUM)
print(f"Anzahl Records: {len(records)}")

Anzahl Records: 18


## Felder und Unterfelder auslesen

Es gibt zwei Arten von Feldern:

- **Kontrollfelder** (`controlfield`) haben einen `tag` und einen Textwert.
- **Datenfelder** (`datafield`) haben einen `tag`, zwei Indikatoren und
  beliebig viele **Unterfelder** (`subfield`) mit einem `code`.

Wir definieren drei kleine Hilfsfunktionen, um bequem an die Werte zu kommen.
`unterfeld()` liest das erste passende Unterfeld, `alle_unterfelder()` sammelt
mehrere Vorkommen (z. B. alle Schlagwörter), und `kontrolle()` liest ein
Kontrollfeld.


In [4]:
def kontrolle(record, tag):
    """Text eines Kontrollfelds (oder None)."""
    feld = record.find(f"m:controlfield[@tag='{tag}']", NAMENSRAUM)
    return feld.text if feld is not None else None


def unterfeld(record, tag, code):
    """Erstes Unterfeld mit diesem Code im ersten Datenfeld mit diesem Tag."""
    feld = record.find(f"m:datafield[@tag='{tag}']", NAMENSRAUM)
    if feld is None:
        return None
    sub = feld.find(f"m:subfield[@code='{code}']", NAMENSRAUM)
    return sub.text if sub is not None else None


def alle_unterfelder(record, tag, code):
    """Alle Unterfelder mit diesem Code aus allen Datenfeldern mit diesem Tag."""
    werte = []
    for feld in record.findall(f"m:datafield[@tag='{tag}']", NAMENSRAUM):
        for sub in feld.findall(f"m:subfield[@code='{code}']", NAMENSRAUM):
            werte.append(sub.text)
    return werte

Testen wir die Funktionen an einem einzelnen Record:

In [5]:
print("Titel:   ", unterfeld(records[0], "245", "a"))
print("Autor:   ", unterfeld(records[0], "100", "a"))
print("Jahr:    ", unterfeld(records[0], "264", "c"))
print("Sprache: ", unterfeld(records[0], "041", "a"))
print("Schlagwörter:", alle_unterfelder(records[0], "650", "a"))

Titel:    Datenkuration in wissenschaftlichen Bibliotheken :
Autor:    Keller, Miriam
Jahr:     2021
Sprache:  ger
Schlagwörter: ['Bibliothekswissenschaft', 'Forschungsdaten', 'Metadaten']


## Alle Records in eine Liste von Dictionaries überführen

Für die Weiterverarbeitung ist es praktisch, jeden Record in ein Dictionary mit
sprechenden Schlüsseln umzuwandeln. Das erinnert an die Arbeit mit
`csv.DictReader` aus dem CSV-Kapitel.


In [6]:
def als_dict(record):
    """Wandelt einen MARC-Record in ein Dictionary um."""
    return {
        "id": kontrolle(record, "001"),
        "isbn": unterfeld(record, "020", "a"),
        "sprache": unterfeld(record, "041", "a"),
        "autor": unterfeld(record, "100", "a"),
        "titel": unterfeld(record, "245", "a"),
        "untertitel": unterfeld(record, "245", "b"),
        "ort": unterfeld(record, "264", "a"),
        "verlag": unterfeld(record, "264", "b"),
        "jahr": unterfeld(record, "264", "c"),
        "schlagwoerter": alle_unterfelder(record, "650", "a"),
    }


titel_liste = [als_dict(record) for record in records]
print(f"{len(titel_liste)} Titel eingelesen")
titel_liste[0]

18 Titel eingelesen


{'id': '000000001',
 'isbn': '9783110470017',
 'sprache': 'ger',
 'autor': 'Keller, Miriam',
 'titel': 'Datenkuration in wissenschaftlichen Bibliotheken :',
 'untertitel': 'Praktiken, Werkzeuge und Standards',
 'ort': 'Berlin',
 'verlag': 'Wissenschaftsverlag Berlin',
 'jahr': '2021',
 'schlagwoerter': ['Bibliothekswissenschaft', 'Forschungsdaten', 'Metadaten']}

## Erscheinungsjahr: 264 und 260

Das Erscheinungsjahr steht je nach Datenalter im Feld `264` (RDA) oder `260`
(AACR2). Eine kleine Funktion prüft beide Felder und liefert das erste
gefundene Jahr.


In [7]:
def erscheinungsjahr(record):
    """Erscheinungsjahr aus 264 oder 260 (oder None)."""
    for tag in ("264", "260"):
        wert = unterfeld(record, tag, "c")
        if wert:
            return wert
    return None


for record in records[:3]:
    print(erscheinungsjahr(record))

2021
2019
2020


## Filter nach Erscheinungsjahr

Da die Jahre als Text gelesen werden, wandeln wir sie mit `int()` in Zahlen um.
Hier filtern wir alle Titel ab dem Jahr 2022.


In [8]:
ab_2022 = [t for t in titel_liste if int(t["jahr"]) >= 2022]
for t in ab_2022:
    print(f"{t['jahr']}  {t['autor']:35} {t['titel']}")

2022  Schneider, Jonas                    Praktische Algorithmen für Bibliotheken :
2023  Nowak, Ewa                          Open Access und die Bibliothek der Zukunft :
2022  Osei, Ama                           Community Archives :
2023  Lindqvist, Elsa                     Biblioteket och AI :
2024  Petrov, Ivan                        Automatisierung mit Python :
2022  Virtanen, Aino                      Open Data for Memory Institutions
2023  Diallo, Fatou                       Katalogisierung nach RDA :


## Filter nach Sprache

Die Sprache steckt als dreistelliger ISO-639-2-Code in Feld `041`, `$a` – etwa
`ger` für Deutsch, `eng` für Englisch oder `fre` für Französisch. Hier zählen
wir die deutschsprachigen Titel.


In [9]:
deutsch = [t for t in titel_liste if t["sprache"] == "ger"]
print(f"Deutschsprachige Titel: {len(deutsch)}")
for t in deutsch:
    print(" -", t["titel"])

Deutschsprachige Titel: 7
 - Datenkuration in wissenschaftlichen Bibliotheken :
 - Praktische Algorithmen für Bibliotheken :
 - Open Access und die Bibliothek der Zukunft :
 - Digitalisierung von Kulturerbe :
 - Digital Humanities in der Praxis :
 - Automatisierung mit Python :
 - Katalogisierung nach RDA :


## Filter nach Schlagwort

Schlagwörter stehen im wiederholbaren Feld `650`. Mit `alle_unterfelder()`
haben wir sie bereits als Liste eingelesen. Nun suchen wir alle Titel, die ein
bestimmtes Schlagwort tragen.


In [10]:
suchwort = "Metadaten"
mit_schlagwort = [t for t in titel_liste if suchwort in t["schlagwoerter"]]
print(f"Titel zum Schlagwort '{suchwort}': {len(mit_schlagwort)}")
for t in mit_schlagwort:
    print(" -", t["titel"])

Titel zum Schlagwort 'Metadaten': 3
 - Datenkuration in wissenschaftlichen Bibliotheken :
 - Digitalisierung von Kulturerbe :
 - Katalogisierung nach RDA :


## Filter kombinieren

Die einzelnen Bedingungen lassen sich mit `and` verknüpfen. Gesucht sind
deutschsprachige Titel ab 2021 mit dem Schlagwort „Metadaten“.


In [11]:
ergebnis = [
    t
    for t in titel_liste
    if t["sprache"] == "ger"
    and int(t["jahr"]) >= 2021
    and "Metadaten" in t["schlagwoerter"]
]
print(f"{len(ergebnis)} Treffer")
for t in ergebnis:
    print(f"{t['jahr']}: {t['titel']}")

2 Treffer
2021: Datenkuration in wissenschaftlichen Bibliotheken :
2023: Katalogisierung nach RDA :


## Zusammenfassung

Sie können jetzt eine MARC-XML-Datei mit `ElementTree` einlesen, Records
iterieren, Felder und Unterfelder über Tag und Code auslesen und die Ergebnisse
filtern. Im nächsten Notebook drehen wir den Weg um: Aus JSON-Quelldaten
erzeugen wir selbst valides MARC-XML.

```{seealso}
Weiter geht es mit [JSON zu MARC-XML](./030-JSON_zu_MARC-XML.ipynb).
```
